In [2]:
import torch                    # Tensor 데이터의 타입으로 파싱하기 위해 로드
import torch.nn as nn           # torch안에 nn부분만 로드하여 nn 별칭으로 사용(nn -> 기본 뼈대)
import torch.optim as optim     # 옵티마이저(기울기의 변화를 주는 기능)

In [3]:
# 데이터셋을 하나 생성
# 독립, 종속 데이터를 tensor로 생성
# 독립 변수 -> sklearn을 이용한 ML에서는 2차원 데이터\
x = torch.tensor( [[1.0] ,[2.0], [3.0],[4.0]])
# 종속 변수 -> sklearn 을 이용한 ML에서는 1차원 -> torch에서는 2차원으로 생성
# 종속 변수는 독립 변수에서 2를 곱하고 1을 더한 값
y = torch.tensor([[3.0],[5.0],[7.0],[9.0]])


In [4]:
type(x)

torch.Tensor

In [5]:
# sklearn에서 모델을 생성한다면 -> 가중치가 2이고 절편이 1인 규칙을 찾는 LinearRegression을 이용하여 예측 가능

# torch을 이용한 단순 선형 회귀
# 순전파 (모델 학습 -> 예측)
    # class 클래스명(부모클래스): --> 부모클래스의 기능을 상속 받아서 클래스를 선언
class LinearReg(nn.Module):

    # torch의 모듈을 이용한 클래스 생성시 2개의 함수를 필요로 선언(생성자 함수, forward 함수)

    # 생성자 함수
    def __init__(self):
        # self : 자기 자신(클래스를 생성할때 저장이 되는 위치)
        # super() : 부모 클래스(nn.Module)
        super(LinearReg, self).__init__()       # 부모 클래스의 생성자 함수를 실행

        # 선형 회귀 모델을 이용
        # nn 모듈 안에 Linear() 모델은
        # 첫번째 인자값 : 입력 데이터(독립 변수)의 차원(피쳐)의 수
        # 두번째 인자값 : 출력 데이터의 차원(피쳐)의 수
        self.linear = nn.Linear(1, 1)
    
    def forward(self, x):
        return self.linear(x)

In [6]:
# 클래스 생성 -> 회귀 모델을 생성
model = LinearReg()

In [7]:
# 손실 함수
criterion = nn.MSELoss()

In [8]:
# 옵티마이저 설정 -> 가중치를 업데이트
# 어떤 모델의 파라미터를 설정할 것인가? -> 첫번째 인자
# lr 매개변수 -> 경사 하강법의 보폭
optimizer = optim.SGD(model.parameters(), lr = 0.01)

In [9]:
# 순전파 (생성된 모델을 호출하려면 -> forward() 함수를 호출하도록 nn.Modul에서 설정이 되어있음)
pred = model(x)
# LinearReg 클래스 안에 forward함수를 호출하여 독립변수(x)를 인자값으로 사용한다

# 손실함수
loss = criterion(pred, y)

# 기울기를 초기화
optimizer.zero_grad()

# 역전파 (자동 미분) -> 데이터가 있는 쪽으로 방향을 제시한다 -> 네비게이션
loss.backward()

# 가중치를 업데이트(파라미터(모델) 수정)
optimizer.step()

# loss 값을 확인
print(loss)

tensor(36.0548, grad_fn=<MseLossBackward0>)


In [10]:
# DL 모델은 반복 학습이 기본 설정 -> 학습모드를 평가모드 전환
# eval() : 모델을 평가모드로 전환
# train() : 모델을 학습모드로 전환
model.eval()

# 예측, 평가 (메모리의 사용량을 줄이기 위해서 가중치의 계산을 잠시 비활성화 )
with torch.no_grad():
    y_pred = model(x)
    loss = criterion(y_pred, y)
    print(y_pred)
    print(loss)

tensor([[0.0223],
        [0.7957],
        [1.5691],
        [2.3425]])
tensor(25.0898)


In [11]:
# 반복 학습을 통해서 가중치와 편향을 변화 시킨다
epochs = 200
model.train()

for epoch in range(epochs):
    # 순전파
    pred = model(x)
    # 손실 함수
    loss = criterion(pred, y)
    # 기울기 초기화
    optimizer.zero_grad()
    # 자동 미분(역전파) -> 가중치의 방향을 제시
    loss.backward()
    # 가중치를 업데이트
    optimizer.step()

    if (epoch + 1) % 20 == 0 :
        # 반복횟수가 20회마다 출력
        print(f'Epoch : [ {epoch+1}, 200 ], Loss : {round(loss.item(), 6)}')

Epoch : [ 20, 200 ], Loss : 0.237017
Epoch : [ 40, 200 ], Loss : 0.188981
Epoch : [ 60, 200 ], Loss : 0.167607
Epoch : [ 80, 200 ], Loss : 0.148664
Epoch : [ 100, 200 ], Loss : 0.131862
Epoch : [ 120, 200 ], Loss : 0.116958
Epoch : [ 140, 200 ], Loss : 0.103739
Epoch : [ 160, 200 ], Loss : 0.092014
Epoch : [ 180, 200 ], Loss : 0.081615
Epoch : [ 200, 200 ], Loss : 0.07239


In [12]:
model.eval()

# 예측, 평가 (메모리의 사용량을 줄이기 위해서 가중치의 계산을 잠시 비활성화 )
with torch.no_grad():
    y_pred = model(x)
    loss = criterion(y_pred, y)
    print(y_pred)
    print(loss)

tensor([[2.5669],
        [4.7901],
        [7.0134],
        [9.2366]])
tensor(0.0720)


In [13]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import numpy as np

In [14]:
data = fetch_california_housing()

x = data['data']
y = data['target']

print(x.shape, y.shape)

(20640, 8) (20640,)


In [15]:
#  1차원 데이터를 2차원으로 변경
y.reshape(-1,1)

array([[4.526],
       [3.585],
       [3.521],
       ...,
       [0.923],
       [0.847],
       [0.894]], shape=(20640, 1))

In [16]:
import pandas as pd
# csv 파일을 이용해서 x,y 생성할때

# df = pd.read_csv('../data/california.csv')

# x = df.drop('target', axis = 1).values
# y = df['target'].values

In [17]:
# 1차원 데이터를 2차원을 변경
y = y.reshape(-1, 1)

In [18]:
# 학습, 평가 데이터로 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [19]:
from sklearn.preprocessing import StandardScaler

In [20]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [21]:
scaler = StandardScaler()
X_train_sc = torch.tensor(scaler.fit_transform(X_train_tensor), dtype=torch.float32)
X_test_sc = torch.tensor(scaler.transform(X_test_tensor), dtype=torch.float32)

In [22]:
X_train_tensor

tensor([[   3.2596,   33.0000,    5.0177,  ...,    3.6918,   32.7100,
         -117.0300],
        [   3.8125,   49.0000,    4.4735,  ...,    1.7381,   33.7700,
         -118.1600],
        [   4.1563,    4.0000,    5.6458,  ...,    2.7232,   34.6600,
         -120.4800],
        ...,
        [   2.9344,   36.0000,    3.9867,  ...,    3.3321,   34.0300,
         -118.3800],
        [   5.7192,   15.0000,    6.3953,  ...,    3.1789,   37.5800,
         -121.9600],
        [   2.5755,   52.0000,    3.4026,  ...,    2.1087,   37.7700,
         -122.4200]])

In [23]:
y_train_tensor

tensor([[1.0300],
        [3.8210],
        [1.7260],
        ...,
        [2.2210],
        [2.8350],
        [3.2500]])

In [24]:
# 선형 회귀 모델 객체를 선언
class Reg(nn.Module):
    # class 생성할때 입력 데이터의 피쳐의 수를 필수 인자로 설정
    def __init__(self, _dim):
        super(Reg, self).__init__()
        self.linear = nn.Linear(_dim, 1)
        
    def forward(self,x):
        return self.linear(x)

In [25]:
# 모델을 생성 -> 생성시 입력 데이터의 피쳐의 개수를 넣어줘야 한다
n_feature = X_train.shape[1]
model = Reg(n_feature)

In [26]:
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.01)

In [27]:
# 반복 실행하면서 모델에 학습하고 가중치 업데이트
epochs = 300

for epoch in range(epochs):
    pred = model(X_train_sc)
    loss = criterion(pred, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    n = epoch + 1

    # 30회마다 loss 확인
    if n % 30 == 0:
        print(f'Epoch : [{n} / 500], Loss : {round(loss.item(), 6)}')

Epoch : [30 / 500], Loss : 2.224694
Epoch : [60 / 500], Loss : 1.156294
Epoch : [90 / 500], Loss : 0.82961
Epoch : [120 / 500], Loss : 0.720718
Epoch : [150 / 500], Loss : 0.677884
Epoch : [180 / 500], Loss : 0.655966
Epoch : [210 / 500], Loss : 0.641279
Epoch : [240 / 500], Loss : 0.629571
Epoch : [270 / 500], Loss : 0.619455
Epoch : [300 / 500], Loss : 0.61043


In [28]:
# 학습된 모델을 이용하여 평가 검증

model.eval()

with torch.no_grad():
    pred = model(X_test_sc)
    loss = criterion(pred, y_test_tensor)

    print(round(loss.item(), 6))

0.622472


In [29]:
for i in range(10):
    print(f'실제 데이터 : {y_test[i]}, 예측 데이터 : {pred[i].item()}')

실제 데이터 : [0.477], 예측 데이터 : 1.0052093267440796
실제 데이터 : [0.458], 예측 데이터 : 1.5638809204101562
실제 데이터 : [5.00001], 예측 데이터 : 2.3346898555755615
실제 데이터 : [2.186], 예측 데이터 : 2.704991340637207
실제 데이터 : [2.78], 예측 데이터 : 2.15566349029541
실제 데이터 : [1.587], 예측 데이터 : 2.1551809310913086
실제 데이터 : [1.982], 예측 데이터 : 2.7195892333984375
실제 데이터 : [1.575], 예측 데이터 : 2.1790173053741455
실제 데이터 : [3.4], 예측 데이터 : 2.0687458515167236
실제 데이터 : [4.466], 예측 데이터 : 4.149280548095703


In [30]:
model2 = Reg(n_feature)
criterion2 = nn.MSELoss()
optimizer2 = optim.SGD(model2.parameters(), lr = 1e-07)

In [31]:
for epoch in range(300):
    pred2 = model2(X_train_tensor)
    loss2 = criterion2(pred2, y_train_tensor)
    optimizer2.zero_grad()
    loss2.backward()
    optimizer2.step()
    n = epoch + 1
    if n % 30 == 0:
        print(loss2)

tensor(614.4064, grad_fn=<MseLossBackward0>)
tensor(569.8759, grad_fn=<MseLossBackward0>)
tensor(528.6984, grad_fn=<MseLossBackward0>)
tensor(490.6219, grad_fn=<MseLossBackward0>)
tensor(455.4127, grad_fn=<MseLossBackward0>)
tensor(422.8541, grad_fn=<MseLossBackward0>)
tensor(392.7469, grad_fn=<MseLossBackward0>)
tensor(364.9059, grad_fn=<MseLossBackward0>)
tensor(339.1604, grad_fn=<MseLossBackward0>)
tensor(315.3526, grad_fn=<MseLossBackward0>)


In [32]:
# 비선형 모델 생성(선형 모델 -> 활성화 함수 -> 선형모델)
class Reg2(nn.Module):
    def __init__ (self,_dim):
        super(Reg2, self).__init__()
        # 다중 퍼셉트론 안에 선형 모델 -> 활성화 함수 -> 선형 모델
        self.model = nn.Sequential(
            # 첫번째 레이어
            nn.Linear(_dim, _dim),
            # 활성화 함수 (비선형 구조 파악) ( Relu(일반적으로 사용), Tanh, Sigmoid)
            nn.ReLU(),
            nn.Linear(_dim, 1)
        )
    def forward(self, x):
        return self.model(x)

In [33]:
model3 = Reg2(n_feature)
criterion3 = nn.MSELoss()
optimizer3 = optim.SGD(model3.parameters(), lr = 0.01)

In [34]:
# 반복 학습
for epoch in range(300):
    n = epoch + 1

    pred3 = model3(X_train_sc)
    loss3 = criterion3(pred3, y_train_tensor)
    optimizer3.zero_grad()
    loss3.backward()
    optimizer3.step()

    if n % 30 == 0:
        print(f'Epoch[{n}/300], Loss : {round(loss3.item(), 6)}')

Epoch[30/300], Loss : 2.363895
Epoch[60/300], Loss : 1.535954
Epoch[90/300], Loss : 1.206617
Epoch[120/300], Loss : 1.002934
Epoch[150/300], Loss : 0.864112
Epoch[180/300], Loss : 0.775115
Epoch[210/300], Loss : 0.719079
Epoch[240/300], Loss : 0.68228
Epoch[270/300], Loss : 0.656348
Epoch[300/300], Loss : 0.637283


In [35]:
model3.eval()
with torch.no_grad():       # 버릇처럼 사용 (메모리 최적화)
    pred3 = model3(X_test_sc)
    loss3 = criterion3(pred3, y_test_tensor)

for i in range(10):
    print(f'실제 데이터 : {y_test[i]}, 예측 데이터 : {pred3[i].item()}')

실제 데이터 : [0.477], 예측 데이터 : 1.273564338684082
실제 데이터 : [0.458], 예측 데이터 : 1.4870535135269165
실제 데이터 : [5.00001], 예측 데이터 : 2.267263889312744
실제 데이터 : [2.186], 예측 데이터 : 2.8318307399749756
실제 데이터 : [2.78], 예측 데이터 : 2.131218910217285
실제 데이터 : [1.587], 예측 데이터 : 2.2443366050720215
실제 데이터 : [1.982], 예측 데이터 : 2.5256857872009277
실제 데이터 : [1.575], 예측 데이터 : 1.9268244504928589
실제 데이터 : [3.4], 예측 데이터 : 2.3180508613586426
실제 데이터 : [4.466], 예측 데이터 : 3.7680110931396484


In [36]:
# 파이토치 랜덤 고정 
torch.manual_seed(42)

In [37]:
# 딥러닝 분류 모델 
import pandas as pd 
from sklearn.metrics import accuracy_score, classification_report

In [38]:
df = pd.read_csv("../csv/iris.csv")
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [39]:
from sklearn.preprocessing import LabelEncoder

In [40]:
# 선형 모델을 이용하여 분류 -> 로지스틱회귀랑 비슷한 방식 
# 출력의 값이 3개의 피쳐로 출력 (  )
x = df.drop('species', axis=1)
y = df['species']

In [41]:
# y의 값들을 LabelEncoder를 이용하여 숫자로 변환 
le = LabelEncoder()
y = le.fit_transform(y)
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [42]:
# train, test 데이터셋으로 분할 
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [43]:
# Sclaer 작업 
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

In [44]:
# Tensor 형태로 변환 
X_train_tensor = torch.tensor(X_train_sc, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_sc, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [45]:
y_test_tensor

tensor([0, 2, 1, 1, 0, 1, 0, 0, 2, 1, 2, 2, 2, 1, 0, 0, 0, 1, 1, 2, 0, 2, 1, 2,
        2, 1, 1, 0, 2, 0])

In [46]:
# 모델 정의 
class clf(nn.Module):
    def __init__(self, _dim):
        super(clf, self).__init__()
        self.model = nn.Linear(_dim, 3)

    def forward(self, x):
        return self.model(x)

In [47]:
clf_model = clf(x.shape[1])

In [48]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(clf_model.parameters(), lr = 0.01)

In [49]:
# for epoch in range(300):

pred = clf_model(X_train_tensor)
pred

tensor([[-8.7668e-01,  1.6183e-01, -1.1046e+00],
        [-3.1913e-01,  1.5752e-01,  4.3691e-01],
        [ 6.2556e-01, -1.8073e-01,  1.2277e+00],
        [-5.2973e-01,  8.9108e-02, -9.9331e-01],
        [-2.5389e-01, -6.5127e-03,  5.8004e-01],
        [ 4.0765e-01, -1.7411e-02,  1.3218e+00],
        [-3.5713e-02, -2.0607e-03,  5.1778e-01],
        [ 4.3783e-01, -1.1812e-01,  1.0508e+00],
        [ 1.6800e+00, -8.5004e-02,  1.9688e+00],
        [ 2.3230e+00, -2.5886e-02,  1.4476e+00],
        [ 1.4802e+00, -2.1836e-01,  2.0149e+00],
        [ 4.3371e-01,  2.6650e-02,  6.6274e-01],
        [-1.0949e+00, -1.0093e-01,  5.6715e-01],
        [-1.1919e-01, -1.3039e-01,  5.6141e-01],
        [-8.4504e-02, -6.1833e-02,  6.4350e-01],
        [-4.7429e-01,  1.6270e-01, -1.1437e+00],
        [ 3.3292e-01,  2.9840e-01, -1.4383e+00],
        [ 4.7656e-01, -2.1898e-01,  1.7175e+00],
        [ 9.1705e-01, -2.7314e-03,  9.9277e-01],
        [ 1.5869e-01,  2.0746e-01, -9.3954e-01],
        [ 1.7239e-01

In [50]:
torch.max(pred, 1)

torch.return_types.max(
values=tensor([0.1618, 0.4369, 1.2277, 0.0891, 0.5800, 1.3218, 0.5178, 1.0508, 1.9688,
        2.3230, 2.0149, 0.6627, 0.5671, 0.5614, 0.6435, 0.1627, 0.3329, 1.7175,
        0.9928, 0.2075, 1.2479, 0.2658, 2.3133, 0.0783, 0.5512, 1.1291, 1.2842,
        0.2460, 0.6946, 0.2714, 0.3145, 0.6495, 0.6456, 1.2248, 1.3334, 1.4135,
        0.4428, 0.2455, 1.6499, 0.8283, 0.2873, 1.0952, 0.6366, 1.9259, 0.1190,
        0.2445, 0.9191, 0.1787, 0.0657, 0.0934, 0.1708, 0.2055, 1.4473, 0.5623,
        0.2058, 0.7130, 0.4636, 1.5064, 0.2034, 0.9191, 0.1896, 0.7443, 1.0183,
        0.1533, 0.9874, 0.5843, 2.2442, 0.3701, 1.0327, 1.3257, 0.1694, 0.2351,
        0.2415, 1.3610, 0.5367, 1.5454, 0.5882, 1.2834, 1.1206, 0.9588, 0.1278,
        0.8986, 1.3429, 0.3459, 1.5273, 0.5364, 2.1302, 0.1067, 0.9258, 0.1270,
        0.6940, 1.2032, 0.3668, 0.2003, 1.4080, 2.4947, 1.3224, 1.2641, 0.9611,
        0.6100, 1.4683, 0.4062, 1.4551, 1.5790, 0.2708, 0.7993, 0.8099, 0.3117,
        0

In [51]:
# 반복 학습 
clf_model.train()
for epoch in range(300):
    pred = clf_model(X_train_tensor)
    loss = criterion(pred, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    n = epoch + 1
    if n % 30 == 0:
        print(f"Epoch : [{n} / 300] , Loss : {round(loss.item(), 6)}")

Epoch : [30 / 300] , Loss : 0.89765
Epoch : [60 / 300] , Loss : 0.771732
Epoch : [90 / 300] , Loss : 0.690923
Epoch : [120 / 300] , Loss : 0.63471
Epoch : [150 / 300] , Loss : 0.592997
Epoch : [180 / 300] , Loss : 0.560507
Epoch : [210 / 300] , Loss : 0.534274
Epoch : [240 / 300] , Loss : 0.512508
Epoch : [270 / 300] , Loss : 0.49406
Epoch : [300 / 300] , Loss : 0.478156


In [52]:
# 평가 
clf_model.eval()

with torch.no_grad():
    pred = clf_model(X_test_tensor)
    _, pred_idx = torch.max(pred, 1)

acc = accuracy_score( y_test, pred_idx )
print("정확도 : ", round(acc, 4))
print(classification_report( y_test, pred_idx ))

정확도 :  0.8
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      0.40      0.57        10
           2       0.62      1.00      0.77        10

    accuracy                           0.80        30
   macro avg       0.88      0.80      0.78        30
weighted avg       0.88      0.80      0.78        30

